# Step2: Permana 2025スタイルの22特徴量への拡張

**前提**: このnotebookの前に、Step1 (MNE tutorialそのまま) のセルを実行し、
`epochs_train` (Alice) と `epochs_test` (Bob) が変数として存在していること。

**このnotebookで追加すること**:
- `src/features.py` に `extract_permana_features()` を追加 (新規関数、既存の`eeg_power_band`は無変更)
- 同じ `RandomForestClassifier(n_estimators=100, random_state=42)` で学習し、Step1と条件を揃えて比較する
- 変更点: 特徴量だけ (5個 → 22個)。分類器・train/testの分け方はStep1と同一。

**新規ライブラリ**: `antropy` (エントロピー計算), `pywavelets` (wavelet分解)


In [ ]:
# 新規ライブラリのインストール (Step2で初めて必要になったもの)
!pip install antropy pywavelets -q


## Step1データの読み込み (このnotebookを単体で実行できるようにするため追加)

**追加理由**: このnotebookを単体で開くと `epochs_train` / `epochs_test` が存在せず
`NameError` になるため。Step1 tutorialのデータ読み込みセルをそのまま (無変更で) ここに追加する。

**注意**: 中身はStep1で実行したコードと完全に同一。`eeg_power_band`の呼び出しはしていない
(Step1notebook側でのみ使う)ため、ここでは読み込みとepoch化までを行う。


In [ ]:
import mne
from mne.datasets.sleep_physionet.age import fetch_data

ALICE, BOB = 0, 1
[alice_files, bob_files] = fetch_data(subjects=[ALICE, BOB], recording=[1])

annotation_desc_2_event_id = {
    "Sleep stage W": 1,
    "Sleep stage 1": 2,
    "Sleep stage 2": 3,
    "Sleep stage 3": 4,
    "Sleep stage 4": 4,
    "Sleep stage R": 5,
}
event_id = {
    "Sleep stage W": 1,
    "Sleep stage 1": 2,
    "Sleep stage 2": 3,
    "Sleep stage 3/4": 4,
    "Sleep stage R": 5,
}

# --- Alice (train) ---
raw_train = mne.io.read_raw_edf(
    alice_files[0], stim_channel="Event marker", infer_types=True,
    preload=True, verbose="error",
)
annot_train = mne.read_annotations(alice_files[1])
annot_train.crop(annot_train[1]["onset"] - 30 * 60, annot_train[-2]["onset"] + 30 * 60)
raw_train.set_annotations(annot_train, emit_warning=False)
events_train, _ = mne.events_from_annotations(
    raw_train, event_id=annotation_desc_2_event_id, chunk_duration=30.0
)
tmax = 30.0 - 1.0 / raw_train.info["sfreq"]
epochs_train = mne.Epochs(
    raw=raw_train, events=events_train, event_id=event_id,
    tmin=0.0, tmax=tmax, baseline=None,
)
del raw_train

# --- Bob (test) ---
raw_test = mne.io.read_raw_edf(
    bob_files[0], stim_channel="Event marker", infer_types=True,
    preload=True, verbose="error",
)
annot_test = mne.read_annotations(bob_files[1])
annot_test.crop(annot_test[1]["onset"] - 30 * 60, annot_test[-2]["onset"] + 30 * 60)
raw_test.set_annotations(annot_test, emit_warning=False)
events_test, _ = mne.events_from_annotations(
    raw_test, event_id=annotation_desc_2_event_id, chunk_duration=30.0
)
epochs_test = mne.Epochs(
    raw=raw_test, events=events_test, event_id=event_id,
    tmin=0.0, tmax=tmax, baseline=None,
)
del raw_test

print(epochs_train)
print(epochs_test)


## `src/features.py` のインポート (GitHub管理版)

`src/features.py` はGitHubリポジトリ (`sleep-xai`) で管理しているため、
`%%writefile`は使わず、リポジトリをcloneしてimportする。

**変更点 (前バージョンからの差分)**:
- 削除: `%%writefile src/features.py` セル
- 追加: `git clone` でリポジトリを取得し、既存の `src/features.py` をそのままimportする方式
- `src/features.py` の中身自体は一切変更していません

⚠️ **リポジトリがprivateのため、このセルを実行する前に一時的にpublicに切り替えてください**
(GitHub → Settings → General → Danger Zone → Change repository visibility)。
実行後はprivateに戻して問題ありません。認証を使ったclone(SSH鍵 / Personal Access Token)は、
必要になったタイミングで改めて提案します。


In [ ]:
# GitHubリポジトリをclone (src/features.py はこの中で管理されている)
# ※ 実行前にリポジトリを一時的にpublicにしておくこと (認証なしでcloneするため)
!git clone https://github.com/naokihayakawa1228/sleep-xai.git
%cd sleep-xai


In [ ]:
import sys
sys.path.append(".")  # clone後にリポジトリのルートに%cd済みなら基本不要だが念のため

from src.features import eeg_power_band, extract_permana_features, PERMANA_FEATURE_NAMES

print("特徴量一覧 (22個):")
print(PERMANA_FEATURE_NAMES)


## 特徴量抽出 (Step1の`epochs_train`, `epochs_test`をそのまま再利用)

`eeg_power_band`をPSD経由で計算していたのに対し、`extract_permana_features`は
生波形からエポックごとにループして計算するため、少し時間がかかります
(841 + 1103エポックで数十秒〜数分程度が目安)。


In [ ]:
X_train_p2 = extract_permana_features(epochs_train, verbose=True)
X_test_p2 = extract_permana_features(epochs_test, verbose=True)

print("X_train_p2.shape =", X_train_p2.shape)
print("X_test_p2.shape  =", X_test_p2.shape)


## 学習・評価 (Step1と同じRandomForest設定で比較)

Step1では `make_pipeline(FunctionTransformer(eeg_power_band), RandomForestClassifier(...))` を
使っていましたが、ここでは既に特徴量抽出済みの `X_train_p2` を直接学習に使います
(FunctionTransformerを介すると`extract_permana_features`が呼ばれるたびに再計算されて遅いため)。
分類器の設定 (`n_estimators=100, random_state=42`) はStep1と完全に揃えています。


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_train = epochs_train.events[:, 2]
y_test = epochs_test.events[:, 2]

clf_p2 = RandomForestClassifier(n_estimators=100, random_state=42)
clf_p2.fit(X_train_p2, y_train)
y_pred_p2 = clf_p2.predict(X_test_p2)

acc_p2 = accuracy_score(y_test, y_pred_p2)
print(f"Step2 (22特徴量) Accuracy: {acc_p2:.3f}")
print(f"(Step1 (5特徴量) Accuracy: 0.685 <- tutorialでの結果)")
print()
print(classification_report(y_test, y_pred_p2, target_names=event_id.keys()))
print(confusion_matrix(y_test, y_pred_p2))


## 確認ポイント

- Step1 (0.685) と比べてAccuracyがどう変化したか
- どのステージのprecision/recallが改善/悪化したか
  (例: Stage 1の分類は難しいことが多いので、そこが伸びたかは要確認)
- 次のStep3 (RF vs XGBoost比較) に進む前に、この22特徴量の結果を記録しておく
